![](img/logo_ucm.jpg)

# Contenerización
## ¿Contenerización SI o NO?

Nos podemos realizar la pregunta, ¿conviene contenerizar un servicio de model serving mediante MLflow (usando Docker)?

Respuesta corta: **Sí, contenerizar tu modelo con MLflow suele ser una buena práctica**, especialmente en entornos productivos o colaborativos.


## ¿Por qué SÍ contenerizar?

| Ventaja              | Explicación                                                                 |
|----------------------|-----------------------------------------------------------------------------|
| **Reproducibilidad** | Te aseguras de que el entorno es el mismo en desarrollo, testing y producción. |
| **Portabilidad**     | Puedes desplegar el modelo en cualquier lugar donde haya Docker (local, cloud, CI/CD, Kubernetes…). |
| **Aislamiento**      | Evitas conflictos de dependencias con otros servicios o librerías del sistema. |
| **Escalabilidad**    | Puedes usar sistemas de orquestación como Kubernetes, ECS, Docker Compose fácilmente. |

>

## ¿Cuándo podrías NO contenerizar?

| Situación               | Justificación                                                                 |
|-------------------------|------------------------------------------------------------------------------|
| **Desarrollo local ligero** | Si estás en etapa exploratoria o demo rápida, puede ser más ágil ejecutar sin Docker. |
| **Entorno controlado**      | Si ya trabajas dentro de un entorno virtual (uv, Conda) bien gestionado y cerrado. |
| **Restricciones de infraestructura** | Si no puedes usar Docker por políticas de la empresa o límites técnicos (por ejemplo en *edge devices*). |

Este último caso suele darse en arquitecturas o entornos que no soportan Docker de forma nativa, como pueden ser los sistemas móviles (iOS/Android), dispositivos embebidos sin un sistema operativo completo (es decir, sin kernel Linux), o situaciones en las que existen problemas de compatibilidad con arquitecturas ARM (como armv7 o arm64), en contraste con la arquitectura x86-64 que utilizan la mayoría de los ordenadores personales.

Un ejemplo práctico sería un modelo de visión por computador desplegado en un dispositivo edge, como una Raspberry Pi. En este caso, el dispositivo no cumplía con los requisitos técnicos para ejecutar Docker de forma eficiente, por limitaciones de memoria/CPU y compatibilidad de arquitectura. Por ello, se optó por realizar la instalación del modelo y sus dependencias directamente en el sistema operativo, sin contenedores, utilizando un entorno virtual ligero.


```{figure} img/industrializacion/robot_aspirador.png
---
name: robot aspirados
width: 50%
align: center
---
Robot aspirador con cámara frontal sobre la cual no se utiliza Docker sino que el modelo se monta directamente sobre la placa base.
```


## Contenerizar un Modelo

### Opción 1: MLflow genera un Dockerfile por defecto

Al guardar el modelo:

```python
mlflow.pyfunc.log_model(..., registered_model_name="my_model")
```

Puedes luego construir una imagen Docker directamente con:

```bash
mlflow models build-docker -m models:/my_model/Production -n my_model_image
```

Y eso genera una imagen lista para servir:

```bash
docker run -p 1234:8080 my_model_image
```

```bash
# Levantamos mlflow en una terminal
mlflow ui --port 5000
# Si tenemos un error similar a mlflow.exceptions.MlflowException: Registered Model with name=Iris_xgboost not found puede que estemos apuntando al puerto incorrecto. Lanza el siguiente comando: 
export MLFLOW_TRACKING_URI=http://localhost:5000 
# Crea la imagen Docker con el siguiente comando, puede llevar unos minutos
mlflow models build-docker -m models:/Iris_xgboost/2 -n iris_xgboost
```


```{figure} img/industrializacion/docker_build.png
---
name: docker build
width: 150%
align: center
---
Docker **build**. Observamos múltiplos stages ejecutando, la primera vez tardará unos minutos, después debido al mecanismo de caching de Docker, solo ejecutará las capas/layers que se hayan modificado.
```


```bash
# Comprobamos que la imagen está creada
docker image ls 
```

```{figure} img/industrializacion/image_ls.png
---
name: imagen ls
width: 80%
align: center
---
 Tamaño total de la imagen iris_xgboost.
```


Una imagen Docker de gran tamaño, como en este caso + de 5 GB, puede generar múltiples inconvenientes: es lenta de descargar, transferir, construir y desplegar, incrementa significativamente el tiempo de arranque de los contenedores, consume espacio en disco y memoria de forma innecesaria, lo cual es especialmente problemático en entornos edge o en la nube. En este caso particular, el tamaño no está justificado, ya que el modelo xgboost tiene un peso reducido y no requiere dependencias pesadas para su inferencia. Aún así, levantemos la imagen: 

```bash
# Levantamos la imagen en otra terminal
docker run -p 1234:8080 iris_xgboost
```


In [3]:
import requests

url = "http://localhost:1234/invocations"
headers = {"Content-Type": "application/json"}

# Datos de entrada en formato dataframe_split
data = {
    "dataframe_split": {
        "columns": ["sepal_length", "sepal_width", "petal_length", "petal_width"],
        "data": [[6.9, 3.1, 5.4, 2.1]]
    }
}

response = requests.post(url=url, headers=headers, json=data)
print("Predicción:", response.json())


Predicción: {'predictions': [2]}


```{figure} img/industrializacion/docker_container_working.png
---
name: docker_container_working
width: 80%
align: center
---
Imagen Docker desplegada como contenedor devolviendo 3 instancias 200 OK
```


```{figure} img/industrializacion/container_up.png
---
name: container_up
width: 150%
align: center
---
Contenedor desplegado y mapeando el puerto 1234 con el interno 8080.
```
NOTA: prueba a lanzar una request con una entrada errónea. 

### Opción 2 (Recomendada): Crear tu propia imagen Docker

Si necesitas personalizar el entorno (añadir FastAPI, NGINX, monitoreo, etc.), puedes usar un Dockerfile propio como este:


```bash
# En la raíz encontramos tanto un main.py como un Dockerfile
docker build -t iris-api .
```

```{figure} img/industrializacion/both_images.png
---
name: both_images
width: 150%
align: center
---
Comparativa de tamaño entre ambas imágenes.
```


```python
# Dentro del main tenemos una variable de entorno que apunta a la variable del modelo que deberemos llamar cuando 
MODEL_URI = os.getenv("MODEL_URI")

```

```bash
# Lanzamos el contenedor
docker run -p 8000:8000 iris-api
```



In [ ]:
import requests

url = "http://127.0.0.1:8000/predict"
headers = {"Content-Type": "application/json"}

data = {
    'sepal_length': 6.9,
    'sepal_width': 3.1,
    'petal_length': 5.4,
    'petal_width': 2.1,
}
response = requests.post(url=url, headers=headers, json=data)
print("Predicción:", response.json())

Predicción: {'prediction': 'virginica'}


## Errores comunes y posibles mejoras


#### 1. Acceso a MLflow para recuperar el/los modelo/s:

Trato de acceder a la ruta file:/app/mlruns/models/Iris_xgboost/version-2 pero obtengo el siguiente error: 

```python
mlflow.exceptions.MlflowException: Could not find an "MLmodel" configuration file at "/app/mlruns/models/Iris_xgboost/version-2/MLmodel"
```

#### Solución corto plazo: 
- Apuntar al experimento concreto: MODEL_URI=file:/app/mlruns/536348599426792075/d61f75314ccd43368b9f28ec5aea7843/artifacts/model

#### Solución largo plazo: 

Desplegar el MLflow server, con host 0.0.0.0 para que potencialmente pudiéramos acceder desde dentro del contenedor. Lanzar los experimentos y registrar el modelo. La base de datos no tiene porqué ser un sqlite, pudiendo utilizar la base de datos coorporativa de la compañía. 

```bash
mlflow server \
  --backend-store-uri sqlite:///mlflow.db \ 
  --default-artifact-root ./mlruns \
  --host 0.0.0.0 \
  --port 5000
```

Añadir la dirección del MLflow tracking server en el main.py: 

```python

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", None)

if MLFLOW_TRACKING_URI:
    import mlflow
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

```

¿Cuál es la URI del servidor?

- Windows/Mac en local podemos utilizar: MLFLOW_TRACKING_URI=http://host.docker.internal:5000
- En linux debemos primero averiguar nuestra IP: 

```bash
ip route | grep default | awk '{print $3}'
# Por ejemplo: 192.168.1.100
```

Para luego añadir la variable al levantar el Docker container: MLFLOW_TRACKING_URI=http://192.168.1.100:5000

>

Por último, en ambos casos podríamos comprobar la conectividad con el servidor haciendo: 

```bash
docker run -it --rm --entrypoint bash iris-api
apt update && apt install -y curl
curl http://192.168.1.100:5000 # Cambiando la IP en el caso de Windows/Mac

```
>

#### 2. Problemas en la gestión de liberías: 

Al incluir directamente los paquetes a instalar en el Dockefile:

```Dockerfile
RUN pip install --no-cache-dir fastapi uvicorn[standard] pandas mlflow scikit-learn xgboost
```

No estamos manteniendo la consistencia ni el versionado entre el entorno de entrenamiento y el de inferencia.

#### Solución:

Utilizar un framework de gestión de dependencias como uv o Conda que nos permita instalar exactamente las mismas librerías usadas durante el desarrollo. 

```Dockerfile

COPY pyproject.toml uv.lock ./

# Instala uv
RUN pip install --no-cache-dir uv

# Instala dependencias desde lock
RUN uv sync --frozen --no-dev

```

Durante estas dos últimas secciones hemos viajado por los diferentes desafíos durante la experimentación con modelos así como su consumo. Sin embargo, todavía no hemos trabajado sobre como unir ambos ecosistemas, experimentación y despligue. En la próxima sección estudiaremos los **patrones de despliegue de modelos** para convertir este ciclo técnico en una operación robusta, escalable y controlada.
